# CTDDG — ChEMBL preprocessing **after Ames RF labelling**

This standalone notebook extracts the ChEMBL-specific logic from **`data_preprocessing.ipynb`** and **`data_preprocessing_1.ipynb`** (original cells 55–70). It operates on the **already RF-labelled** ChEMBL CSV; it does **not** retrain or run a Random Forest.

**What is retained from the originals:** ≤100 atoms; salt/fragment processing; removing ChEMBL molecules overlapping with BindingDB ligands (train **and** test); canonical deduplication; and atom-type vocabulary `(symbol, formal_charge, explicit_H_count)` from ChEMBL + BindingDB.

**Necessary fixes:** preserve `Ames_prediction`; use RDKit structures, not SMILES text length, to select fragments; canonicalize before comparisons; deterministic atom-type ordering for newly trained models; enforce CTDDG's 120-step limit and four supported bond types; report every filtering decision.

**Outputs** under `OUTPUT_DIR`:

- `chembl_preprocessed_labelled.csv` — `Smiles,Ames_prediction` (keeps predictions).
- `chembl_final.txt` — one `SMILES 0/1` per line, the training format used by the original CTDDG pretraining loader.
- `chembl.txt` — SMILES-only text file matching the old preprocessing notebook's output.
- `atom_types.txt` — graph atom vocabulary, preserving an existing vocabulary's order if supplied.
- `preprocessing_report.json` — source paths, settings, counts, and label distribution.

**Important:** `0/1` labels are predicted **Ames mutagenicity**, not experimental ground truth or universal toxicity. The actual input was produced after RF labelling; this notebook must not invent or change its labels.

## 1 — File paths and policy settings

Edit `LABELLED_CHEMBL_CSV` and **one** BindingDB option:

- Set `BINDINGDB_ROOT` to a directory containing the five train/five test ligand files from the original preprocessing notebook, **or**
- Set `BINDINGDB_LIGAND_FILES` to a list of already prepared ligand text/CSV files (for instance a single complete BindingDB ligand collection).

BindingDB overlap removal is required by default: the notebook **stops** if no files are found. This is intentional to avoid accidentally creating a contaminated pretraining set.

If you will **resume an existing CTDDG checkpoint**, set `EXISTING_ATOM_TYPES` to that model's exact `atom_types.txt`: the order determines atom indices. Otherwise leave it `None` to build a sorted new vocabulary for training **from scratch**.

In [ ]:
from pathlib import Path

# INPUTS: EDIT THESE
LABELLED_CHEMBL_CSV = Path('/home/muhamed/repo/CTDDG/REWORKED_CODE/Random_Forest/rf_outputs/chembl_pretrain_labelled.csv')
BINDINGDB_ROOT = Path('/home/muhamed/repo/CTDDG/datasets/data/bindingdb')
BINDINGDB_LIGAND_FILES = []  # Optional explicit paths; overrides root discovery if nonempty
EXISTING_ATOM_TYPES = None  # Or Path('datasets/data/atom_types.txt') for a pretrained checkpoint

# OUTPUTS
OUTPUT_DIR = Path('/home/muhamed/repo/CTDDG/REWORKED_CODE/chembl_preprocessed')
OVERWRITE_OUTPUTS = False

# MOLECULAR FILTERS AND GRAPH COMPATIBILITY
MAX_ATOMS = 100
MAX_GENERATION_STEPS = 120  # CTDDG uses number_of_bonds + 1
CHUNK_SIZE = 25_000
MAX_INPUT_ROWS = None      # Keep None for FULL dataset; small integer only for test run

# PREVIOUS RF labelling notebook used largest heavy-atom fragment.
# If unexpected disconnected SMILES remain here, skip by default rather than silently
# changing the molecule whose RF label was predicted. Can opt in if verified.
ALLOW_FRAGMENT_CHANGES = False

# IMPORTANT: The notebook's new vocabulary is suitable for NEW training only.
# When loading any existing model, provide its exact vocabulary in EXISTING_ATOM_TYPES.

print('Labelled ChEMBL:', LABELLED_CHEMBL_CSV)
print('BindingDB root:', BINDINGDB_ROOT)
print('Output directory:', OUTPUT_DIR)

## 2 — RDKit and other imports

RDKit performs molecule parsing, largest-fragment selection, canonicalization and graph-compatibility checks. Run `%pip install rdkit` in a separate cell only if the import is missing; then restart the kernel.

In [ ]:
import csv
import json
import os
import shutil
import time
from collections import Counter

import pandas as pd
from IPython.display import display
from rdkit import Chem, RDLogger, rdBase

RDLogger.DisableLog('rdApp.error')
print('RDKit:', rdBase.rdkitVersion, '| Pandas:', pd.__version__)

## 3 — Discover BindingDB ligand files

The originals read five `d*_tr_ligands.txt` and five `d*_te_ligands.txt` files, created by preprocessing the original BindingDB train/test files. This notebook looks for all ten; if none of those ten are present, it also supports the original `ligands_train_4_org_corrected` / `ligands_test_4_org_corrected` names. Explicit file paths take priority.

For fidelity, **all five train and five test groups** must be present when using automatic discovery. An explicit file list is accepted as a user-specified complete ligand collection.

In [ ]:
FOLD_NAMES = ['4_org_1042_104', '4_org_1000_112', '4_org_1004_122',
              '4_org_1002_103', '4_org_1036_124']

def discover_bindingdb_files(root, explicit):
    if explicit:
        paths = [Path(p) for p in explicit]
        missing = [str(p) for p in paths if not p.is_file()]
        if missing:
            raise FileNotFoundError('BindingDB file(s) missing: ' + ', '.join(missing))
        return paths

    if not root.is_dir():
        raise FileNotFoundError(
            f'BindingDB directory not found: {root}. Edit BINDINGDB_ROOT '
            'or set BINDINGDB_LIGAND_FILES to your complete ligand file(s).')

    # Prefer the already processed ten ligand files.
    processed = []
    for i in range(1, 6):
        for split in ('tr', 'te'):
            matches = sorted(root.rglob(f'd{i}_{split}_ligands.txt'))
            if matches:
                processed.append(matches[0])
    if len(processed) == 10:
        return processed
    if processed:
        raise ValueError(f'Found only {len(processed)}/10 processed BindingDB fold files. '
                         'Provide all files (or a complete explicit collection).')

    raw = []
    for fold in FOLD_NAMES:
        for split in ('train', 'test'):
            pattern = f'ligands_{split}_4_org_corrected'
            matches = sorted(p for p in root.rglob(pattern) if fold in str(p))
            if matches:
                raw.append(matches[0])
    if len(raw) == 10:
        return raw
    raise FileNotFoundError(
        f'Found {len(raw)}/10 original BindingDB fold ligand files under {root}. '
        'Supply the original five train + five test files, or explicitly set '
        'BINDINGDB_LIGAND_FILES to a complete consolidated BindingDB ligand source.')

bindingdb_files = discover_bindingdb_files(BINDINGDB_ROOT, BINDINGDB_LIGAND_FILES)
print(f'BindingDB input files: {len(bindingdb_files)}')
for p in bindingdb_files:
    print(' -', p)

## 4 — Shared molecule normalization and graph rules

- Reject missing or invalid SMILES.
- If a molecule has disconnected fragments, select the fragment with most **heavy atoms** (not the longest text representation).
- Convert to RDKit canonical isomeric SMILES.
- Require at most 100 RDKit atoms, at most 120 generation steps (`bonds + 1`), and only `AROMATIC`, `SINGLE`, `DOUBLE`, or `TRIPLE` bonds.

**Labels stay attached to their molecular structures.** The labelled ChEMBL CSV should already have been fragment-cleaned by the RF notebook. If a ChEMBL row would now change its molecular components (not just its SMILES spelling), we reject it by default and count it rather than assume its earlier RF label still applies.

In [ ]:
ALLOWED_BOND_TYPES = {
    Chem.BondType.AROMATIC, Chem.BondType.SINGLE,
    Chem.BondType.DOUBLE, Chem.BondType.TRIPLE,
}

def prepare_molecule(value, allow_fragment_changes=True):
    if pd.isna(value) or not str(value).strip():
        return None, None, 'missing_smiles', False
    mol = Chem.MolFromSmiles(str(value).strip())
    if mol is None or mol.GetNumAtoms() == 0:
        return None, None, 'invalid_smiles', False

    fragments = Chem.GetMolFrags(mol, asMols=True, sanitizeFrags=True)
    fragment_changed = len(fragments) > 1
    if fragment_changed:
        if not allow_fragment_changes:
            return None, None, 'unexpected_fragments', True
        mol = max(fragments, key=lambda m: (m.GetNumHeavyAtoms(), m.GetNumAtoms(),
                                           Chem.MolToSmiles(m, canonical=True)))

    if mol.GetNumAtoms() > MAX_ATOMS:
        return None, None, 'too_many_atoms', fragment_changed
    if mol.GetNumBonds() + 1 > MAX_GENERATION_STEPS:
        return None, None, 'too_many_steps', fragment_changed
    if any(b.GetBondType() not in ALLOWED_BOND_TYPES for b in mol.GetBonds()):
        return None, None, 'unsupported_bond', fragment_changed

    smi = Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)
    return smi, mol, None, fragment_changed

def atom_types_in_mol(mol):
    return {(a.GetSymbol(), a.GetFormalCharge(), a.GetNumExplicitHs())
            for a in mol.GetAtoms()}

def parse_binary_label(v):
    try:
        n = float(v)
    except (ValueError, TypeError):
        return None
    if n not in (0.0, 1.0):
        return None
    return int(n)

print('Preprocessing rules initialized.')

## 5 — Build BindingDB overlap set and extract its atom types

Read BindingDB ligand files **without proteins** (we only need molecular structures for overlap exclusion and atom vocabulary). The original notebook separately did salt removal and atom-count filtering on BindingDB; we replicate those molecular restrictions with safer RDKit parsing.

The overlap set includes all valid canonicalized BindingDB ligand structures, even if a ligand later fails a graph-size filter. Its atom types are collected only from molecules compatible with our CTDDG graph rules.

In [ ]:
def iter_ligands_from_file(path):
    path = Path(path)
    if path.suffix.lower() in ('.csv', '.tsv'):
        delimiter = '\t' if path.suffix.lower() == '.tsv' else ','
        with path.open('r', encoding='utf-8-sig', errors='replace') as f:
            reader = csv.DictReader(f, delimiter=delimiter)
            if not reader.fieldnames:
                raise ValueError(f'Empty CSV: {path}')
            col = next((k for k in reader.fieldnames if k.strip().lower() in
                        ('smiles', 'canonical_smiles', 'canonical smiles', 'smile', 'ligand_smiles')), None)
            if col is None:
                raise KeyError(f'No SMILES column in {path}; found {reader.fieldnames}')
            for row in reader:
                yield row.get(col)
    else:
        with path.open('r', encoding='utf-8', errors='replace') as f:
            for line in f:
                line = line.strip()
                if line:
                    # Original *_ligands*.txt files have one SMILES per line.
                    yield line.split()[0]

bindingdb_smiles_set = set()
bindingdb_atom_types = set()
bindingdb_stats = Counter()
start = time.monotonic()

for path in bindingdb_files:
    for s in iter_ligands_from_file(path):
        bindingdb_stats['input_ligands'] += 1
        # First canonicalize valid BindingDB compounds for overlap checks.
        # Do NOT require size or bonds <= max for membership in the overlap set.
        raw = Chem.MolFromSmiles(str(s)) if s is not None else None
        if raw is None or raw.GetNumAtoms() == 0:
            bindingdb_stats['invalid'] += 1
            continue
        frags = Chem.GetMolFrags(raw, asMols=True, sanitizeFrags=True)
        selected = max(frags, key=lambda m: (m.GetNumHeavyAtoms(), m.GetNumAtoms(),
                                             Chem.MolToSmiles(m, canonical=True)))
        key = Chem.MolToSmiles(selected, canonical=True, isomericSmiles=True)
        bindingdb_smiles_set.add(key)
        if selected.GetNumAtoms() <= MAX_ATOMS and selected.GetNumBonds() + 1 <= MAX_GENERATION_STEPS:
            if all(b.GetBondType() in ALLOWED_BOND_TYPES for b in selected.GetBonds()):
                bindingdb_atom_types.update(atom_types_in_mol(selected))
                bindingdb_stats['compatible_ligands'] += 1

print(f'BindingDB ligand records: {bindingdb_stats["input_ligands"]:,}')
print(f'Unique canonical overlap keys: {len(bindingdb_smiles_set):,}')
print(f'BindingDB atom types: {len(bindingdb_atom_types):,}')
print(f'Elapsed: {time.monotonic()-start:.1f} s')

## 6 — Validate input and existing atom vocabulary

If an existing atom vocabulary is specified, preserve its original line order exactly and exclude ChEMBL structures that contain atom types not present in it. This prevents atom-index mismatches with existing checkpoints.

If starting from scratch, assemble a deterministic vocabulary from the surviving ChEMBL molecules plus graph-compatible BindingDB ligands.

In [ ]:
if not LABELLED_CHEMBL_CSV.is_file():
    raise FileNotFoundError(f'Labelled ChEMBL input not found: {LABELLED_CHEMBL_CSV}')

with LABELLED_CHEMBL_CSV.open('r', encoding='utf-8-sig') as f:
    first_line = f.readline()
CSV_SEPARATOR = max([',', ';', '\t'], key=lambda d: first_line.count(d))
columns = list(pd.read_csv(LABELLED_CHEMBL_CSV, sep=CSV_SEPARATOR, nrows=0).columns)
smiles_col = next((c for c in columns if c.lower().strip() == 'smiles'), None)
label_col = next((c for c in columns if c.lower().strip() == 'ames_prediction'), None)
if smiles_col is None or label_col is None:
    raise KeyError('Expected labelled ChEMBL CSV columns Smiles,Ames_prediction. Found: '
                   + str(columns))

existing_vocab_lines = None
existing_vocab_set = None
if EXISTING_ATOM_TYPES is not None:
    path = Path(EXISTING_ATOM_TYPES)
    if not path.is_file():
        raise FileNotFoundError(f'Existing atom vocabulary missing: {path}')
    existing_vocab_lines = [line.strip() for line in path.read_text().splitlines() if line.strip()]
    existing_vocab_set = set()
    for line in existing_vocab_lines:
        parts = line.split(',')
        if len(parts) != 3:
            raise ValueError(f'Bad atom_types.txt entry: {line!r}')
        existing_vocab_set.add((parts[0], int(parts[1]), int(parts[2])))
    if len(existing_vocab_lines) != len(existing_vocab_set):
        raise ValueError('Existing atom_types.txt has duplicate entries; check it before use.')
    print('Existing vocabulary loaded WITHOUT REORDERING:', len(existing_vocab_lines), 'atom types')
else:
    print('No existing vocabulary — will generate a deterministic new vocabulary.')

print('Input columns:', columns)
print('SMILES column:', smiles_col, '| label column:', label_col)
print('Detected separator:', repr(CSV_SEPARATOR))

## 7 — Process labelled ChEMBL (streamed by chunks)

This is the principal step. Order of operations:

1. Validate `Ames_prediction` (0 or 1) and molecule.
2. Apply graph size/bond rules, keep or reject fragmented SMILES according to the setting.
3. Canonicalize, remove BindingDB overlap, reject atom-vocabulary outliers if using an existing vocabulary.
4. Remove duplicate canonical molecules. **If duplicate structures carry conflicting labels, stop with an error** rather than choosing one arbitrarily.
5. Write the CSV, SMILES-only TXT, and `SMILES label` TXT. A temporary-file strategy prevents partial results from masquerading as completed outputs.

This processing does **not** re-run the RF. It assumes the labels were generated on the canonical, fragment-cleaned molecules output by the preceding RF notebook.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
outputs = {
    'labelled_csv': OUTPUT_DIR / 'chembl_preprocessed_labelled.csv',
    'training_txt': OUTPUT_DIR / 'chembl_final.txt',
    'smiles_txt': OUTPUT_DIR / 'chembl.txt',
    'atom_types_txt': OUTPUT_DIR / 'atom_types.txt',
    'report_json': OUTPUT_DIR / 'preprocessing_report.json',
}
existing_outputs = [str(p) for p in outputs.values() if p.exists()]
if existing_outputs and not OVERWRITE_OUTPUTS:
    raise FileExistsError('Output file(s) already exist. Set OVERWRITE_OUTPUTS=True only if you '
                          'want to replace them. Existing: ' + ', '.join(existing_outputs))

temporary = {k: p.with_name(p.name + '.partial') for k,p in outputs.items()}
if any(p.exists() for p in temporary.values()):
    raise FileExistsError('Old .partial files found. Inspect and remove them before re-running: '
                          + ', '.join(str(p) for p in temporary.values() if p.exists()))

chembl_stats = Counter()
label_counts = Counter()
seen_labels = {}
chembl_atom_types = set()
start = time.monotonic()

try:
    with temporary['labelled_csv'].open('w', newline='', encoding='utf-8') as out_csv, \
         temporary['training_txt'].open('w', encoding='utf-8') as out_train, \
         temporary['smiles_txt'].open('w', encoding='utf-8') as out_smiles:
        writer = csv.writer(out_csv)
        writer.writerow(['Smiles', 'Ames_prediction'])

        rows_processed = 0
        for chunk_idx, chunk in enumerate(pd.read_csv(
                LABELLED_CHEMBL_CSV, sep=CSV_SEPARATOR,
                usecols=[smiles_col, label_col], chunksize=CHUNK_SIZE,
                dtype={smiles_col:'string', label_col:'string'}), 1):
            for raw_smi, raw_label in zip(chunk[smiles_col], chunk[label_col]):
                if MAX_INPUT_ROWS is not None and rows_processed >= MAX_INPUT_ROWS:
                    break
                rows_processed += 1
                chembl_stats['input_rows'] += 1
                label = parse_binary_label(raw_label)
                if label is None:
                    chembl_stats['invalid_label'] += 1
                    continue
                smi, mol, reason, changed = prepare_molecule(
                    raw_smi, allow_fragment_changes=ALLOW_FRAGMENT_CHANGES)
                if reason is not None:
                    chembl_stats[reason] += 1
                    continue
                if changed:
                    chembl_stats['fragments_stripped'] += 1
                if smi in bindingdb_smiles_set:
                    chembl_stats['bindingdb_overlap'] += 1
                    continue
                atoms = atom_types_in_mol(mol)
                if existing_vocab_set is not None and not atoms.issubset(existing_vocab_set):
                    chembl_stats['out_of_vocabulary_atom'] += 1
                    continue
                if smi in seen_labels:
                    if seen_labels[smi] != label:
                        raise ValueError('CONFLICTING PREDICTIONS for the same canonical molecule: '
                                         f'{smi} -> {seen_labels[smi]} and {label}. '
                                         'Do not arbitrarily keep one; investigate label provenance.')
                    chembl_stats['duplicate'] += 1
                    continue

                seen_labels[smi] = label
                chembl_atom_types.update(atoms)
                label_counts[label] += 1
                chembl_stats['kept'] += 1
                writer.writerow([smi, label])
                out_train.write(f'{smi} {label}\n')
                out_smiles.write(f'{smi}\n')

            if chunk_idx == 1 or chunk_idx % 5 == 0:
                print(f'Chunks {chunk_idx:,} | processed={chembl_stats["input_rows"]:,} '
                      f'| kept={chembl_stats["kept"]:,} '
                      f'| elapsed={time.monotonic()-start:.1f}s')
            if MAX_INPUT_ROWS is not None and rows_processed >= MAX_INPUT_ROWS:
                break

    # Create the vocabulary: original order if provided, sorted new union otherwise.
    if existing_vocab_lines is not None:
        final_vocab_lines = existing_vocab_lines
    else:
        union = chembl_atom_types | bindingdb_atom_types
        final_vocab_lines = [f'{sym},{charge},{hs}' for sym,charge,hs in sorted(union)]
    with temporary['atom_types_txt'].open('w', encoding='utf-8') as out:
        for line in final_vocab_lines:
            out.write(line + '\n')

    if chembl_stats['kept'] == 0:
        raise ValueError('No ChEMBL molecules survived; check file paths, labels and filters.')

    report = {
        'labelled_chembl_input': str(LABELLED_CHEMBL_CSV.resolve()),
        'bindingdb_inputs': [str(p.resolve()) for p in bindingdb_files],
        'existing_atom_types': str(Path(EXISTING_ATOM_TYPES).resolve()) if EXISTING_ATOM_TYPES is not None else None,
        'settings': {'max_atoms': MAX_ATOMS, 'max_generation_steps': MAX_GENERATION_STEPS,
                     'allow_fragment_changes': ALLOW_FRAGMENT_CHANGES,
                     'max_input_rows': MAX_INPUT_ROWS, 'chunk_size': CHUNK_SIZE},
        'bindingdb_counts': dict(bindingdb_stats),
        'bindingdb_unique_overlap_keys': len(bindingdb_smiles_set),
        'chembl_counts': dict(chembl_stats),
        'output_label_counts': {str(k): v for k,v in sorted(label_counts.items())},
        'atom_vocab_count': len(final_vocab_lines),
        'note': 'Ames_prediction is model-predicted mutagenicity, not an experimental ChEMBL label.',
    }
    temporary['report_json'].write_text(json.dumps(report, indent=2) + '\n', encoding='utf-8')
    for key, dest in outputs.items():
        os.replace(temporary[key], dest)
except Exception:
    for path in temporary.values():
        path.unlink(missing_ok=True)
    raise

print('\nComplete. Kept:', chembl_stats['kept'], '| label counts:', dict(label_counts))
print('Atom types:', len(final_vocab_lines), '| elapsed:', round(time.monotonic()-start, 1), 's')
for key, dest in outputs.items():
    print(f'{key}: {dest.resolve()}')

## 8 — Verify finished outputs and review filtering report

Read back the output rows and check alignment across all three molecule files and both label classes. The first five records give a quick visual check without loading the whole dataset into memory again.

In [ ]:
report = json.loads(outputs['report_json'].read_text(encoding='utf-8'))
assert report['chembl_counts']['kept'] == sum(int(v) for v in report['output_label_counts'].values())
preview = pd.read_csv(outputs['labelled_csv'], nrows=5)
assert list(preview.columns) == ['Smiles', 'Ames_prediction']
assert set(preview['Ames_prediction'].dropna().astype(int)).issubset({0, 1})
with outputs['training_txt'].open() as f:
    first_training_lines = [f.readline().strip() for _ in range(5)]
for row, line in zip(preview.itertuples(index=False), first_training_lines):
    assert line == f'{row.Smiles} {int(row.Ames_prediction)}'

print('Preprocessed ChEMBL preview:')
display(preview)
print('Pretraining format (SMILES label):')
print('\n'.join(first_training_lines))
print('\nMolecule filtering counts:')
display(pd.DataFrame(sorted(report['chembl_counts'].items()), columns=['reason','count']))
print('\nOutput predicted label counts:', report['output_label_counts'])
print('\nAtom vocabulary size:', report['atom_vocab_count'])

## Using these files for CTDDG

- Use `chembl_final.txt` if your pretraining loader accepts the original **`SMILES label`** format. In the original `pretraining_CTDDG.ipynb`, `process_single()` splits lines on a space to read the label. Check your **current PyTorch** implementation uses the same contract before running a long training job.
- Use `atom_types.txt` only **with matching model atom indices**. A new sorted vocabulary is suitable for training a *new* model, **not** for swapping into an already-trained checkpoint.
- `chembl.txt` is provided for backward compatibility but **has no toxicity labels**. Do not use it as the training input to a toxicity-conditioned model.
- The original notebooks do **not** prepare explicit model graph tensors in this ChEMBL section; that graph-building belongs to the CTDDG data loader.
- Because the earlier RF notebook may already have canonicalized, removed fragments, removed duplicate molecules, and applied size checks, some rejection counts may be zero here. That is fine.
- **BindingDB overlap is excluded** from pretraining by molecular identity after canonicalization; identical molecules expressed differently in SMILES are caught, but stereochemistry, tautomerism, or other chemically related structures are not treated as identical unless the canonical SMILES matches.